In [24]:
import openai, json

client = openai.OpenAI()
messages = []

In [25]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "인기 영화 목록을 조회합니다.",
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_detail",
            "description": "id로 영화 상세 정보 조회",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "상세하게 조회하고 싶은 영화의 id"
                    }
                },
                "required": ["id"],
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "유사한 영화 조회",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "유사한 영화를 조회할 대상 영화의 id"
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credit",
            "description": "영화의 출연진 및 제작진 조회",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "출연진 및 제작진을 조회할 영화의 id"
                    }
                },
                "required": ["id"]
            }
        }
    }
]

In [26]:
import requests

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    return json.dumps(response.json())

def get_movie_detail(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    return json.dumps(response.json())

def get_similar_movies(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/similar")
    return json.dumps(response.json())

def get_movie_credit(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/credits")
    return json.dumps(response.json())

FUNCTION_MAP = {
    'get_popular_movies': get_popular_movies,
    'get_movie_detail': get_movie_detail,
    'get_similar_movies': get_similar_movies,
    'get_movie_credit': get_movie_credit
}

In [27]:
from openai.types.chat import ChatCompletionMessage

def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS
    )

    process_ai_response(response.choices[0].message)

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role":"assistant", 
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments
                        }
                    } 
                    for tool_call in message.tool_calls
                ]
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            function_arguments = tool_call.function.arguments
            print(f"[{function_name}({function_arguments}) 호출")

            try:
                function_arguments = json.loads(function_arguments)
            except json.JSONDecodeError:
                function_arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**function_arguments)

            # print(f"Ran {function_name} with args {function_arguments} for a result of {result}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": result
            })

        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"LLM : {message.content}")

In [28]:
while True:
    message = input("User Input")
    if message == "quit":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User : {message}")
        call_ai()

User : 지금 인기 있는 영화 알려줘
[get_popular_movies({}) 호출
LLM : 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Shelter**
   - 개요: 자발적으로 유배당한 남자가 폭풍에서 어린 소녀를 구하면서 그의 과거와 관련된 적들로부터 그녀를 보호하기 위해 고립에서 벗어나게 되는 사건이 벌어집니다.
   - 개봉일: 2026-01-28
   - 평점: 6.8
   - [포스터 보기](https://image.tmdb.org/t/p/w780/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg)

2. **The Bluff**
   - 개요: 고요한 삶을 살고 있던 섬에서 복수를 꿈꾸는 전 선장이 돌아오면서 과거의 잔혹한 기억을 다시 마주하게 되는 이야기입니다.
   - 개봉일: 2026-02-17
   - 평점: 6.0
   - [포스터 보기](https://image.tmdb.org/t/p/w780/sojEzvfxR2DBcDSJyAisX8TWjov.jpg)

3. **Mercy**
   - 개요: 미래의 한 형사가 아내를 살해한 혐의로 재판에 서게 되고, 자신을 변호하기 위해 고급 AI 판사가 결정하기 전에 진실을 밝혀내야 하는 긴박한 상황을 그립니다.
   - 개봉일: 2026-01-20
   - 평점: 7.1
   - [포스터 보기](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

4. **Hellfire**
   - 개요: 신비한 과거를 가진 방랑자가 작은 마을에 도착하면서 범죄 보스와 맞서 싸워야 하는 이야기입니다.
   - 개봉일: 2026-02-05
   - 평점: 7.1
   - [포스터 보기](https://image.tmdb.org/t/p/w780/tQti9QTf13MfzNpXguijgNh7ojE.jpg)

5. **Your Heart Will Be Broken**
   - 개요: 고등학생 폴리나가 새로운 학교에서 괴롭힘으로부터